# Re-score saved Gemma detections
Re-parses the saved `response` field in `predictions.jsonl` using the fixed label aliases and recomputes P/R/F1/mAP without rerunning Gemma.

In [27]:
# ---- CONFIG ----
DATASET = "simd"
M = 3
SAVE_DIR = f"/home/gridsan/manderson/ovdsat/run/eval/detection/{DATASET}_label_desc/gemma/gemma-4-E4B-it/M{M}"
PREDICTIONS_FILE = f"{SAVE_DIR}/predictions.jsonl"
ANNOTATIONS_FILE = f"/home/gridsan/manderson/ovdsat/data/{DATASET}/val_coco-{M}-label-desc.json"
SAVE_RESULTS = f"{SAVE_DIR}/results_reparsed.txt"
SAVE_PREDICTIONS = f"{SAVE_DIR}/predictions_reparsed.jsonl"

In [28]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np


def normalize_label(label):
    return re.sub(r"[^a-z0-9]+", "", str(label).lower())


def strip_leading_article(label):
    return re.sub(r"^\s*(?:a|an|the)\s+", "", str(label).strip(), flags=re.IGNORECASE)


def get_category_aliases(name, dataset):
    aliases = set()

    def add(alias):
        alias = str(alias).strip().strip(" .;:")
        if alias:
            aliases.add(normalize_label(alias))

    name = str(name).strip()
    no_article = strip_leading_article(name)
    parts = name.split(",", 1)
    short_name = parts[0].strip()
    short_no_article = strip_leading_article(short_name)
    description = parts[1].strip() if len(parts) == 2 else None

    # Full text + article-free text + short label before first comma.
    add(name)
    add(no_article)
    add(short_name)
    add(short_no_article)

    if description:
        add(description)
        add(strip_leading_article(description))

    # MAR: "a C-17 aircraft with ..." -> also accept "C-17 aircraft" and "C-17".
    if dataset.lower() == "mar":
        m = re.match(r"^(.+?\baircraft)\b", no_article, flags=re.IGNORECASE)
        if m:
            aircraft_name = m.group(1).strip()
            add(aircraft_name)
            add(re.sub(r"\s+aircraft$", "", aircraft_name, flags=re.IGNORECASE))

    return aliases


def extract_json(text):
    match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if match:
        return json.loads(match.group(1))

    start = text.find("[")
    end = text.rfind("]")
    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("No JSON array found")


def parse_predictions(text, width, height, label_to_class):
    try:
        raw_predictions = extract_json(text)
    except Exception:
        return np.zeros((0, 6), dtype=np.float32), set()

    predictions = []
    unknown_labels = set()

    for item in raw_predictions:
        if not isinstance(item, dict) or "box_2d" not in item or "label" not in item:
            continue

        label = normalize_label(item["label"])
        if label not in label_to_class:
            unknown_labels.add(str(item["label"]))
            continue

        try:
            y1, x1, y2, x2 = map(float, item["box_2d"])
        except (TypeError, ValueError):
            continue

        x1 = np.clip(x1 / 1000.0 * width, 0, width)
        y1 = np.clip(y1 / 1000.0 * height, 0, height)
        x2 = np.clip(x2 / 1000.0 * width, 0, width)
        y2 = np.clip(y2 / 1000.0 * height, 0, height)
        x1, x2 = sorted((x1, x2))
        y1, y2 = sorted((y1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        predictions.append([x1, y1, x2, y2, 1.0, label_to_class[label]])

    return np.asarray(predictions, dtype=np.float32).reshape(-1, 6), unknown_labels


def box_iou(box1, box2):
    if len(box1) == 0 or len(box2) == 0:
        return np.zeros((len(box1), len(box2)), dtype=np.float32)

    inter_x1 = np.maximum(box1[:, None, 0], box2[None, :, 0])
    inter_y1 = np.maximum(box1[:, None, 1], box2[None, :, 1])
    inter_x2 = np.minimum(box1[:, None, 2], box2[None, :, 2])
    inter_y2 = np.minimum(box1[:, None, 3], box2[None, :, 3])

    inter_w = np.clip(inter_x2 - inter_x1, 0, None)
    inter_h = np.clip(inter_y2 - inter_y1, 0, None)
    inter = inter_w * inter_h

    area1 = np.clip(box1[:, 2] - box1[:, 0], 0, None) * np.clip(box1[:, 3] - box1[:, 1], 0, None)
    area2 = np.clip(box2[:, 2] - box2[:, 0], 0, None) * np.clip(box2[:, 3] - box2[:, 1], 0, None)

    return inter / (area1[:, None] + area2[None, :] - inter + 1e-16)


def process_batch(detections, labels, iouv):
    correct = np.zeros((len(detections), len(iouv)), dtype=bool)
    if len(detections) == 0 or len(labels) == 0:
        return correct

    iou = box_iou(labels[:, 1:], detections[:, :4])
    correct_class = labels[:, 0:1] == detections[:, 5][None, :]

    for i, threshold in enumerate(iouv):
        gt_idx, pred_idx = np.where((iou >= threshold) & correct_class)
        if len(gt_idx) == 0:
            continue

        matches = np.column_stack([gt_idx, pred_idx, iou[gt_idx, pred_idx]])
        matches = matches[np.argsort(-matches[:, 2])]
        _, keep = np.unique(matches[:, 1], return_index=True)
        matches = matches[np.sort(keep)]
        matches = matches[np.argsort(-matches[:, 2])]
        _, keep = np.unique(matches[:, 0], return_index=True)
        matches = matches[np.sort(keep)]

        correct[matches[:, 1].astype(int), i] = True

    return correct


def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    x = np.linspace(0.0, 1.0, 101)
    y = np.interp(x, mrec, mpre)
    return np.sum((x[1:] - x[:-1]) * (y[1:] + y[:-1]) * 0.5)


def calculate_class_metrics(correct, confidence, predicted_classes, target_classes, num_classes):
    niou = correct.shape[1]
    p = np.zeros(num_classes)
    r = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    ap = np.zeros((num_classes, niou))

    predicted_classes = predicted_classes.astype(int)
    target_classes = target_classes.astype(int)

    for class_id in range(num_classes):
        pred_indices = np.where(predicted_classes == class_id)[0]
        num_targets = int(np.sum(target_classes == class_id))

        if len(pred_indices) == 0 or num_targets == 0:
            continue

        order = np.argsort(-confidence[pred_indices], kind="stable")
        class_correct = correct[pred_indices][order].astype(float)

        tp = np.cumsum(class_correct, axis=0)
        fp = np.cumsum(1.0 - class_correct, axis=0)
        recall_curve = tp / (num_targets + 1e-16)
        precision_curve = tp / (tp + fp + 1e-16)

        p[class_id] = precision_curve[-1, 0]
        r[class_id] = recall_curve[-1, 0]
        if p[class_id] + r[class_id] > 0:
            f1[class_id] = 2 * p[class_id] * r[class_id] / (p[class_id] + r[class_id])

        for j in range(niou):
            ap[class_id, j] = compute_ap(recall_curve[:, j], precision_curve[:, j])

    return p, r, f1, ap[:, 0], ap.mean(axis=1)


In [29]:
# Load COCO annotations and build the corrected label -> class mapping.
with open(ANNOTATIONS_FILE, "r", encoding="utf-8") as f:
    coco = json.load(f)

categories = sorted(coco["categories"], key=lambda c: c["id"])
names = [c["name"] for c in categories]
category_id_to_class = {c["id"]: i for i, c in enumerate(categories)}

label_to_class = {}
for class_id, name in enumerate(names):
    for alias in get_category_aliases(name, DATASET):
        label_to_class[alias] = class_id

images_by_id = {img["id"]: img for img in coco["images"]}
annotations_by_image = defaultdict(list)
for ann in coco["annotations"]:
    if not ann.get("iscrowd", 0) and ann["category_id"] in category_id_to_class:
        annotations_by_image[ann["image_id"]].append(ann)

print(f"{len(coco['images'])} images, {len(coco['annotations'])} annotations, {len(names)} classes")
print(f"{len(label_to_class)} accepted normalized label aliases")


4121 images, 36764 annotations, 15 classes
88 accepted normalized label aliases


In [30]:
# Reparse every saved model response and recompute IoU matches.
rows = []
with open(PREDICTIONS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

iouv = np.linspace(0.5, 0.95, 10)
stats = []
reparsed_rows = []
unknown_counts = defaultdict(int)
old_prediction_count = 0
new_prediction_count = 0

for row in rows:
    image_id = row["image_id"]
    image_info = images_by_id[image_id]
    width, height = image_info["width"], image_info["height"]

    # Use the saved parsed model response: NO model inference.
    response = row.get("response", "")
    predictions, unknown = parse_predictions(response, width, height, label_to_class)

    for label in unknown:
        unknown_counts[label] += 1

    old_prediction_count += len(row.get("predictions", []))
    new_prediction_count += len(predictions)

    gt_boxes = []
    gt_classes = []
    for ann in annotations_by_image.get(image_id, []):
        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue
        gt_boxes.append([x, y, x + w, y + h])
        gt_classes.append(category_id_to_class[ann["category_id"]])

    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    gt_classes = np.asarray(gt_classes, dtype=np.float32)

    labels = (
        np.column_stack([gt_classes, gt_boxes])
        if len(gt_classes)
        else np.zeros((0, 5), dtype=np.float32)
    )
    correct = process_batch(predictions, labels, iouv)

    stats.append((
        correct,
        predictions[:, 4] if len(predictions) else np.zeros(0, dtype=np.float32),
        predictions[:, 5] if len(predictions) else np.zeros(0, dtype=np.float32),
        gt_classes,
    ))

    reparsed = dict(row)
    reparsed["predictions"] = predictions.tolist()
    reparsed_rows.append(reparsed)

print(f"Saved rows: {len(rows)}")
print(f"Predictions accepted before: {old_prediction_count}")
print(f"Predictions accepted now:    {new_prediction_count}")
print(f"Still-unrecognized labels:   {sum(unknown_counts.values())}")


Saved rows: 4121
Predictions accepted before: 1
Predictions accepted now:    17365
Still-unrecognized labels:   1261


In [31]:
# Compute the same per-class and overall metrics as eval_gemma_detection.py.
correct = np.concatenate([s[0] for s in stats], axis=0)
confidence = np.concatenate([s[1] for s in stats], axis=0)
predicted_classes = np.concatenate([s[2] for s in stats], axis=0)
target_classes_all = np.concatenate([s[3] for s in stats], axis=0)

nc = len(names)
nt = np.bincount(target_classes_all.astype(int), minlength=nc)

class_p, class_r, class_f1, class_ap50, class_ap5095 = calculate_class_metrics(
    correct, confidence, predicted_classes, target_classes_all, nc
)

target_classes = np.where(nt > 0)[0]
if len(target_classes):
    mp = class_p[target_classes].mean()
    mr = class_r[target_classes].mean()
    mf1 = class_f1[target_classes].mean()
    map50 = class_ap50[target_classes].mean()
    map5095 = class_ap5095[target_classes].mean()
else:
    mp = mr = mf1 = map50 = map5095 = 0.0

class_width = max(len("Class"), len("all"), max(map(len, names)))
header = (
    f"{'Class':<{class_width}} {'Images':>8} {'Instances':>10} "
    f"{'P':>10} {'R':>10} {'F1':>10} {'mAP50':>10} {'mAP50-95':>10}"
)
lines = [header]
lines.append(
    f"{'all':<{class_width}} {len(rows):>8d} {int(nt.sum()):>10d} "
    f"{mp:>10.4g} {mr:>10.4g} {mf1:>10.4g} {map50:>10.4g} {map5095:>10.4g}"
)

for i, name in enumerate(names):
    lines.append(
        f"{name:<{class_width}} {len(rows):>8d} {int(nt[i]):>10d} "
        f"{class_p[i]:>10.4g} {class_r[i]:>10.4g} {class_f1[i]:>10.4g} "
        f"{class_ap50[i]:>10.4g} {class_ap5095[i]:>10.4g}"
    )

results_text = "\n".join(lines) + "\n"
print(results_text)

Path(SAVE_RESULTS).write_text(results_text, encoding="utf-8")
with open(SAVE_PREDICTIONS, "w", encoding="utf-8") as f:
    for row in reparsed_rows:
        f.write(json.dumps(row) + "\n")

print(f"Saved metrics:     {SAVE_RESULTS}")
print(f"Saved reparsed:    {SAVE_PREDICTIONS}")


Class                                                                                                                                                                 Images  Instances          P          R         F1      mAP50   mAP50-95
all                                                                                                                                                                     4121      36764     0.2461     0.1785     0.1476     0.1751    0.09222
a car, a small road vehicle with four wheels and a compact body                                                                                                         4121      17611     0.4511     0.3071     0.3655        0.3     0.1332
a truck, a large road vehicle with a cargo bed or box-shaped rear section                                                                                               4121       2222       0.17     0.1512     0.1601     0.1039    0.05029
an airliner, a large passenger aircraft with

In [45]:
# Optional diagnostic: labels that still do not map to a dataset class.
for label, count in sorted(unknown_counts.items(), key=lambda x: -x[1])[:30]:
    print(f"{count:5d}  {label}")


  314  terrain
  121  roadway
   67  expressway
   46  building
   39  truck
   37  railway facility with tracks
   21  a building
   17  a road transport such as a car, truck, bus, or van
   15  an expressway
   14  road
   13  a swimming pool
   12  bases
   11  a roadway
    7  runway
    6  water
    6  parking lot
    6  a pier
    5  center markings
    5  a parking lot
    4  terminal
    3  a central athletic field
    3  roadside complex with parking
    3  industrial facility
    3  a railway facility with tracks
    3  a house
    2  aircraft parking areas
    2  train
    2  pier
    2  taxiway
    2  a truck
